In [ ]:
import pandas as pd
import os

In [ ]:
def load_and_prep_nutrition_matrix(data_dir):
    print("Iniciando pipeline de carga de datos FDC...")
    
    # 1. Cargar solo las columnas necesarias para optimizar memoria
    print("Cargando archivos CSV...")
    df_food = pd.read_csv(os.path.join(data_dir, 'food.csv'), usecols=['fdc_id', 'description'])
    
    df_nutrient = pd.read_csv(os.path.join(data_dir, 'nutrient.csv'), usecols=['id', 'name', 'unit_name'])
    
    df_food_nutrient = pd.read_csv(os.path.join(data_dir, 'food_nutrient.csv'), 
                                   usecols=['fdc_id', 'nutrient_id', 'amount'])
    
    df_portion = pd.read_csv(os.path.join(data_dir, 'food_portion.csv'), 
                             usecols=['fdc_id', 'amount', 'measure_unit_id', 'modifier', 'gram_weight'])
                             
    df_measure_unit = pd.read_csv(os.path.join(data_dir, 'measure_unit.csv'), usecols=['id', 'name'])

    # 2. Definir los nutrientes de interés (Filtro de dimensionalidad)
    # IDs estándar en FDC: 1008 (Energía), 1003 (Proteína), 1004 (Grasas), 1005 (Carbohidratos)
    target_nutrients = [1008, 1003, 1004, 1005] 
    
    # Filtrar solo los nutrientes objetivo
    df_fn_filtered = df_food_nutrient[df_food_nutrient['nutrient_id'].isin(target_nutrients)].copy()
    
    # Regla de Calidad de Datos: Eliminar duplicados para evitar errores en el Pivot
    # (A veces FDC tiene múltiples análisis para el mismo nutriente en el mismo alimento)
    df_fn_filtered = df_fn_filtered.drop_duplicates(subset=['fdc_id', 'nutrient_id'])

    # 3. Construir la Matriz A (Pivot)
    print("Pivoteando matriz de nutrientes...")
    nutrition_matrix = df_fn_filtered.pivot(index='fdc_id', columns='nutrient_id', values='amount').reset_index()
    
    # Renombrar columnas para mayor legibilidad
    nutrient_names = df_nutrient.set_index('id')['name'].to_dict()
    nutrition_matrix = nutrition_matrix.rename(columns=nutrient_names)

    # 4. Unir con las descripciones de los alimentos
    print("Integrando descripciones...")
    final_df = pd.merge(df_food, nutrition_matrix, on='fdc_id', how='inner')

    # 5. Unir con las porciones (para pasar de 100g a medidas reales)
    # Nos quedamos con la primera porción disponible por alimento para simplificar la matriz inicial
    df_portion_unique = df_portion.drop_duplicates(subset=['fdc_id']).copy()
    
    # Traducir el ID de la unidad (ej. "cup", "slice")
    df_portion_unique = pd.merge(df_portion_unique, df_measure_unit, 
                                 left_on='measure_unit_id', right_on='id', how='left')
    
    # Unir la porción al DataFrame final
    final_df = pd.merge(final_df, df_portion_unique[['fdc_id', 'amount', 'name', 'modifier', 'gram_weight']], 
                        on='fdc_id', how='left')
    
    # Renombrar columnas de porciones para claridad
    final_df = final_df.rename(columns={
        'amount': 'portion_amount',
        'name': 'portion_unit'
    })

    # 6. Limpieza final (Drop NaNs en macronutrientes críticos para mantener factibilidad matemática)
    critical_cols = [nutrient_names[id] for id in target_nutrients if id in nutrient_names]
    final_df = final_df.dropna(subset=critical_cols)
    
    # Rellenar con 100g aquellos alimentos que no tengan una porción definida en la base de datos
    final_df['gram_weight'] = final_df['gram_weight'].fillna(100.0)
    final_df['portion_unit'] = final_df['portion_unit'].fillna('g')
    final_df['portion_amount'] = final_df['portion_amount'].fillna(100.0)

    print(f"Pipeline completado. Matriz final: {final_df.shape[0]} alimentos y {final_df.shape[1]} columnas.")
    return final_df

In [ ]:
# --- Ejecución ---
if __name__ == "__main__":
    # Ajusta esta ruta a la carpeta donde tienes tus CSVs
    directorio_datos = './foundation_food' 
    
    try:
        matriz_optimizacion = load_and_prep_nutrition_matrix(directorio_datos)
        
        # Mostrar las primeras filas y estructura
        print("\nVista previa de la Matriz de Optimización:")
        print(matriz_optimizacion.head())
        
    except FileNotFoundError as e:
        print(f"\nError: No se encontró la ruta de los datos. Verifica el directorio. Detalles: {e}")